# Variance Inflation Factor (VIF)

We’ll calculate VIF to detect multicollinearity.

Formula: VIF = 1 / (1 - R²) for each feature.

Rule: If VIF > 10, drop that variable.

aa


When you are building regression models, your features are supposed to be like a well-coordinated team where everyone has a unique role. But sometimes, two or more features are essentially doing the exact same job—they are highly correlated.

In statistics, this "clashing" is called **Multicollinearity**, and the **Variance Inflation Factor (VIF)** is the ultimate diagnostic tool to detect it.

## 1. What is VIF and Its Purpose?

**Variance Inflation Factor (VIF)** is a metric that measures how much the variance (or uncertainty) of an estimated regression coefficient is increased, or "inflated," due to collinearity with other features in the model.

### The Purpose:

Its primary purpose is to identify **redundant features** so you can remove them. When features are highly correlated, the regression model struggles to figure out which feature is actually driving the target variable. It's like having two people shouting the exact same instructions at you at the exact same time—it becomes impossible to tell who is contributing what. VIF pinpoint-identifies these "shouters."

## 2. The Statistical Concept Behind It

To understand VIF, you have to look at how it is calculated. Surprisingly, VIF is calculated by running regression models *on your features themselves*, completely ignoring your actual target variable ($Y$).

For every single independent feature ($X_i$) in your dataset, the VIF calculation does the following:

1. It treats $X_i$ as a temporary "target variable."
2. It treats all *other* remaining features ($X_1, X_2, \dots, X_k$) as predictors.
3. It fits an ordinary least squares regression: $$ X_i = $\beta_0$ + $\beta_1 X_1$ + $\beta_2 X_2$ + $\dots$ + e 
4. It calculates the $R^2$ value of this model (let's call it $R^2_i$). This $R^2_i$ represents how well all the other features can predict $X_i$.
5. It plugs $R^2_i$ into the VIF formula:
$$

\text{VIF}_i = \frac{1}{1 - R^2_i}
$$

### The Intuition behind the Math:

- **No Correlation:** If the other features cannot predict $X_i$ at all, $R^2_i = 0$. The formula gives $\text{VIF} = 1 / (1 - 0) = 1$. This is the absolute baseline (no collinearity).
- **High Correlation:** If other features perfectly predict $X_i$, $R^2_i = 0.90$ (90% of its variance is explained by others). The formula gives $\text{VIF} = 1 / (1 - 0.90) = 10$. Your variance is now inflated tenfold!

## 3. How to Interpret VIF Values

When you run a VIF analysis, you will get a score for every single feature:

| VIF Value | Status | What it means | Action |
| --- | --- | --- | --- |
| **$\text{VIF} = 1$** | **Perfect** | No correlation at all with other features. | Keep. |
| **$1 \lt \text{VIF} \lt 5$** | **Moderate** | Tolerable correlation. | Generally safe to keep. |
| **$\text{VIF} \ge 5$ or $10$** | **High** | Severe multicollinearity. The feature is highly redundant. | **Candidate for removal.** |

> ⚠️ *Note: While some strict statisticians use $5$ as the cutoff, $10$ is the most common industry-standard threshold for flagging problematic multicollinearity.*
>
>

## 4. When to Use and When Not

### When to Use:

- **Before running Linear or Logistic Regression:** These models are highly sensitive to multicollinearity. High VIFs lead to wild swings in your coefficient estimates with tiny changes in data.
- **When interpreting feature coefficients is critical:** If you need to tell your business stakeholders, *"Every 1-unit increase in Feature A causes a $50 increase in sales,"* you **must** ensure Feature A has a low VIF. Otherwise, that "$50" estimate is highly unstable and unreliable.

### When NOT to Use:

- **When using purely predictive, non-linear ML models:** Algorithms like Random Forests, Gradient Boosting (XGBoost), or Neural Networks do not care about multicollinearity in the same way linear models do. They can handle highly correlated features without their predictive performance collapsing.
- **For control variables or polynomial terms:** If you purposefully added a squared term to your model (like $X$ and $X^2$ to capture a curve), they will naturally have a massive VIF. You should ignore high VIFs for these intentionally constructed relationships.
- **For Dummy (One-Hot) Encoded Categorical Variables:** If you encode a variable with many categories, they can naturally show high VIFs. This is often benign and doesn't always require intervention.

## 5. Key Details and Pitfalls to Be Aware Of

- **The "Standard Error" Inflation:** The actual standard error of a feature's coefficient is multiplied by $\sqrt{\text{VIF}}$. If a feature has a VIF of 9, its standard error is tripled ($\sqrt{9} = 3$). This makes your confidence intervals 3 times wider, and makes it highly likely that the feature's $p$-value will balloon, making a truly important feature look "statistically insignificant."
- **Don't drop all high-VIF features at once!** This is a classic beginner mistake. If Feature A and Feature B are highly correlated with each other, *both* will show a massive VIF (e.g., 15). If you drop both, you lose valuable data. Instead:
1. Find the feature with the absolute highest VIF.
2. Drop **only** that one feature.
3. Re-calculate the VIFs for the remaining features. You will often find that dropping the worst offender instantly collapses the VIFs of the other correlated features back down to safe levels!
- **Alternatives to dropping:** If you cannot afford to lose either of two highly correlated features, you can combine them (e.g., averaging them, or using Principal Component Analysis to merge them into a single component), or switch to regularized regression methods like **Ridge or Lasso regression**, which are mathematically designed to handle collinearity.

In [19]:
import pandas as pd

# Import pandas so we can read and work with tabular data.
# The read_csv() function loads the training dataset from disk into a DataFrame.
# A DataFrame is a table-like structure that makes data analysis easier.
df = pd.read_csv("D:/KrishNaikDataScience/LiveProjects/Level4_featureEngg/26_Apr_FeatureEngg_3/house-prices-advanced-regression-techniques/train.csv")

# Display the first few rows to quickly inspect the dataset structure and columns.
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [20]:
### Step 1: Select only numerical columns
df_numeric = df.select_dtypes(include=['number'])

### Step 2 & 3: Check for missing values and drop columns with >30% missing
total_rows = len(df_numeric)
missing_counts = df_numeric.isnull().sum()
missing_percentage = (missing_counts / total_rows) * 100

columns_to_drop = missing_percentage[missing_percentage > 30].index.tolist()
df_cleaned = df_numeric.drop(columns=columns_to_drop)

print("Dropping columns with >30% missing values:")
print(columns_to_drop)
print("-" * 30)

df_cleaned.info()

Dropping columns with >30% missing values:
[]
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   OverallQual    1460 non-null   int64  
 5   OverallCond    1460 non-null   int64  
 6   YearBuilt      1460 non-null   int64  
 7   YearRemodAdd   1460 non-null   int64  
 8   MasVnrArea     1452 non-null   float64
 9   BsmtFinSF1     1460 non-null   int64  
 10  BsmtFinSF2     1460 non-null   int64  
 11  BsmtUnfSF      1460 non-null   int64  
 12  TotalBsmtSF    1460 non-null   int64  
 13  1stFlrSF       1460 non-null   int64  
 14  2ndFlrSF       1460 non-null   int64  
 15  LowQualFinSF   1460 non-null   int64  
 16  GrLivArea      1460 non-null  

In [21]:
for col in df_cleaned.columns:
    if df_cleaned[col].isnull().sum() > 0:
        mean_val = df_cleaned[col].mean()
        # Fix the warning by reassigning the column
        df_cleaned[col] = df_cleaned[col].fillna(mean_val)
        print(f"Imputed missing values in '{col}' with the mean ({mean_val}).")
print("-" * 30)

Imputed missing values in 'LotFrontage' with the mean (70.04995836802665).
Imputed missing values in 'MasVnrArea' with the mean (103.68526170798899).
Imputed missing values in 'GarageYrBlt' with the mean (1978.5061638868744).
------------------------------


In [22]:
import statsmodels.api as sm

# Create the feature matrix by removing the target column from the dataset.
# We want to analyze the independent variables, so SalePrice is excluded.
X = df_cleaned.drop(columns=["SalePrice"])

# Add a constant column (intercept) required by statsmodels regression models.
# This helps the model estimate the baseline value when all predictors are zero.
X = sm.add_constant(X)

In [23]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Import the VIF function from statsmodels.
# This function computes how much a feature's variance is inflated due to multicollinearity.

In [24]:
vif_data = pd.DataFrame()

# Create an empty DataFrame that will store the feature names and their VIF scores.
# This structure makes the results easy to read and inspect.

In [25]:
vif_data["Feature"] = X.columns

# Store the column names of the feature matrix in a new column called Feature.
# These are the variables for which VIF will be calculated.

In [26]:
X.shape[1]

# Return the number of columns in X, which is the number of features.
# This tells us how many VIF values will be computed.

38

In [27]:
X.values

array([[1.000e+00, 1.000e+00, 6.000e+01, ..., 0.000e+00, 2.000e+00,
        2.008e+03],
       [1.000e+00, 2.000e+00, 2.000e+01, ..., 0.000e+00, 5.000e+00,
        2.007e+03],
       [1.000e+00, 3.000e+00, 6.000e+01, ..., 0.000e+00, 9.000e+00,
        2.008e+03],
       ...,
       [1.000e+00, 1.458e+03, 7.000e+01, ..., 2.500e+03, 5.000e+00,
        2.010e+03],
       [1.000e+00, 1.459e+03, 2.000e+01, ..., 0.000e+00, 4.000e+00,
        2.010e+03],
       [1.000e+00, 1.460e+03, 2.000e+01, ..., 0.000e+00, 6.000e+00,
        2.008e+03]], shape=(1460, 38))

In [29]:
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# i denotes the index of the feature for which we are calculating the VIF.

# Compute a VIF score for each feature in X.
# The list comprehension loops over every column index from 0 to number_of_features - 1.
# For each feature, variance_inflation_factor() evaluates how strongly it is explained by the other features.
# A high VIF means the feature is highly correlated with other predictors and may cause multicollinearity issues.

In [30]:
print(f"VIF (House Price Dataset) : ")
print(vif_data)

# Print the full VIF table so the results are visible in the output.
# This helps us inspect which features have strong multicollinearity.

high_vif = vif_data[vif_data["VIF"] > 10]

# Filter the DataFrame to keep only features whose VIF is greater than 10.
# This is a common threshold used to flag serious multicollinearity.

if not high_vif.empty:
    print(f"High multicollinearity detected in these features :")
    print(high_vif)

# If any features exceed the threshold, print them for review.
# If the filtered result is empty, no highly problematic features were found.

VIF (House Price Dataset) : 
          Feature           VIF
0           const  2.416585e+06
1              Id  1.026804e+00
2      MSSubClass  1.657110e+00
3     LotFrontage  1.571589e+00
4         LotArea  1.256787e+00
5     OverallQual  3.265164e+00
6     OverallCond  1.596343e+00
7       YearBuilt  5.029641e+00
8    YearRemodAdd  2.425296e+00
9      MasVnrArea  1.399007e+00
10     BsmtFinSF1           inf
11     BsmtFinSF2           inf
12      BsmtUnfSF           inf
13    TotalBsmtSF           inf
14       1stFlrSF           inf
15       2ndFlrSF           inf
16   LowQualFinSF           inf
17      GrLivArea           inf
18   BsmtFullBath  2.219624e+00
19   BsmtHalfBath  1.153285e+00
20       FullBath  2.951113e+00
21       HalfBath  2.168248e+00
22   BedroomAbvGr  2.329674e+00
23   KitchenAbvGr  1.597118e+00
24   TotRmsAbvGrd  4.889626e+00
25     Fireplaces  1.586110e+00
26    GarageYrBlt  3.370938e+00
27     GarageCars  5.578949e+00
28     GarageArea  5.468729e+00
29     Wood